Problem statement: A financial institution wants to predict whether a customer will default on a loan before approving it. Early identification of risky customers helps reduce financial loss.
You are working as a Machine Learning Analyst and must build a classification model using the K-Nearest Neighbors (KNN) algorithm to predict loan default.
This case introduces:
Mixed feature types
Financial risk interpretation
Class imbalance awareness

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Use the below data set to predict the loan default
data = {
    'Age':[28,45,35,50,30,42,26,48,38,55],
    'AnnualIncome':[6.5,12,8,15,7,10,5.5,14,9,16],
    'CreditScore':[720,680,750,640,710,660,730,650,700,620],
    'LoanAmount':[5,10,6,12,5,9,4,11,7,13],
    'LoanTerm':[5,10,7,15,5,10,4,12,8,15],
    'EmploymentType':['Salaried','Self-Employed','Salaried',
                      'Self-Employed','Salaried','Salaried',
                      'Salaried','Self-Employed','Salaried',
                      'Self-Employed'],
    'LoanDefault':[0,1,0,1,0,1,0,1,0,1]
}

df = pd.DataFrame(data)

print(df)

   Age  AnnualIncome  CreditScore  LoanAmount  LoanTerm EmploymentType  \
0   28           6.5          720           5         5       Salaried   
1   45          12.0          680          10        10  Self-Employed   
2   35           8.0          750           6         7       Salaried   
3   50          15.0          640          12        15  Self-Employed   
4   30           7.0          710           5         5       Salaried   
5   42          10.0          660           9        10       Salaried   
6   26           5.5          730           4         4       Salaried   
7   48          14.0          650          11        12  Self-Employed   
8   38           9.0          700           7         8       Salaried   
9   55          16.0          620          13        15  Self-Employed   

   LoanDefault  
0            0  
1            1  
2            0  
3            1  
4            0  
5            1  
6            0  
7            1  
8            0  
9            1 

Enclode categorical variable:
Encoding: Salaried = 0
Self-Employed = 1

In [ ]:
encoder = LabelEncoder()

df['EmploymentType'] = encoder.fit_transform(
    df['EmploymentType']
)
print(df)

   Age  AnnualIncome  CreditScore  LoanAmount  LoanTerm  EmploymentType  \
0   28           6.5          720           5         5               0   
1   45          12.0          680          10        10               1   
2   35           8.0          750           6         7               0   
3   50          15.0          640          12        15               1   
4   30           7.0          710           5         5               0   
5   42          10.0          660           9        10               0   
6   26           5.5          730           4         4               0   
7   48          14.0          650          11        12               1   
8   38           9.0          700           7         8               0   
9   55          16.0          620          13        15               1   

   LoanDefault  
0            0  
1            1  
2            0  
3            1  
4            0  
5            1  
6            0  
7            1  
8            0  
9   

Define Feature and Target

In [ ]:
X = df.drop('LoanDefault', axis=1)

y = df['LoanDefault']

Train vs Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Feature Scaling: KNN uses distance calculations.
Therefore scaling is mandatory.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Build KNN Model**

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)

knn.fit(X_train_scaled,y_train)

KNeighborsClassifier(n_neighbors=3)

**Predictions**

In [ ]:
y_pred = knn.predict(X_test_scaled)

print("Predictions:", y_pred)

Predictions: [0 1]


**Model Evaluation**

In [ ]:
print("Accuracy:",
      accuracy_score(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy: 1.0

Confusion Matrix
[[1 0]
 [0 1]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



**Predict New Customer Risk**

In [ ]:
new_customer = pd.DataFrame({
    'Age':[40],
    'AnnualIncome':[8],
    'CreditScore':[670],
    'LoanAmount':[10],
    'LoanTerm':[10],
    'EmploymentType':[1]   # Self-Employed
})

new_customer_scaled = scaler.transform(new_customer)

prediction = knn.predict(new_customer_scaled)

print("Prediction[0]: ", prediction[0])

if prediction[0] == 1:
    print("High Risk - Loan Default Likely")
else:
    print("Low Risk - Loan Default Unlikely")


Prediction[0]:  1
High Risk - Loan Default Likely


**Find Best K Value**

In [ ]:
accuracy = []

for k in range(1,8):
    model = KNeighborsClassifier(n_neighbors=k)

    model.fit(X_train_scaled, y_train)

    pred = model.predict(X_test_scaled)

    accuracy.append(
        accuracy_score(y_test, pred)
    )

for k, acc in zip(range(1,8), accuracy):
    print("K =", k, " Accuracy =", acc)

K = 1  Accuracy = 1.0
K = 2  Accuracy = 1.0
K = 3  Accuracy = 1.0
K = 4  Accuracy = 1.0
K = 5  Accuracy = 1.0
K = 6  Accuracy = 1.0
K = 7  Accuracy = 1.0


1. Identify High-Risk Customers:
Customers having:
Lower Credit Score, Higher Loan Amount, Longer Loan Term, Self-Employed status, Moderate income relative to loan size are classified as high risk.
Examples:
1) Age 45, CreditScore 680, Loan 10 → Default
2) Age 50, CreditScore 640, Loan 12 → Default
3) Age 55, CreditScore 620, Loan 13 → Default

2. What Patterns Lead to Loan Default?
Observed patterns for Defaulters:
Credit Score: 620-680
Loan Amount: 9-13 lakhs
Loan Term: 10-15 years
Mostly Self-Employed

Non-Defaulters:
Credit Score: 700-750
Loan Amount: 4-7 lakhs
Loan Term: 4-8 years
Mostly Salaried

Pattern for default:
Higher Default Risk = Low Credit Score + High Loan Amount + Long Loan Duration

3. How Do Credit Score and Income Influence Predictions?
Credit Score:
Strong inverse relationship:
Higher Credit Score → Lower Risk
Lower Credit Score → Higher Risk

Annual Income
Higher income generally improves repayment capacity.
Ex. Income 6.5 lakh, Loan 5 lakh → Safe
Income 12 lakh, Loan 10 lakh → Riskier because of larger loan

4. Suggested Banking Policies
Policy 1:
Auto-approve:
Credit Score > 700, Loan Amount < 7 Lakhs, Stable Employment

Policy 2:
Require guarantor or collateral if:
Credit Score < 650 and Large Loan Amount

5. Compare KNN with Decision Trees:
| Feature | KNN | Decision Tree |
|----------|----------|----------|
| Model Type | Distance-based | Rule-based |
| Training Speed | Fast | Fast |
| Prediction Speed | Slower | Faster |
| Interpretability | Low | High |
| Handles Nonlinearity | Yes | Yes |
| Scaling Required | Yes | No |
| Business Explanation | Difficult | Easy |

Recommendation For banking:
Decision Trees are often preferred because they provide clear approval/rejection rules.
Example:
IF CreditScore < 650, AND LoanAmount > 10 THEN Default = Yes

6. What Happens if LoanAmount Dominates Distance Calculation?
Suppose features are not scaled.
Example:
CreditScore = 720
LoanAmount = 5
Credit score range: 300-900
Loan amount range:4-13
One feature may dominate distance measurements and distort neighbor selection.
Consequences:
Wrong nearest neighbors, Poor predictions, Reduced accuracy
Solution:
Always standardize data before using KNN.

In [ ]:
from sklearn.preprocessing import StandardScaler

7. Should KNN Be Used in Real-Time Loan Approval Systems?
Advantages: Easy to implement, Works well on small datasets, Captures nonlinear relationships
Limitations: Slow prediction for millions of records, Requires feature scaling. Difficult to explain decisions, Sensitive to noise and outliers

Real Banking Recommendation
Use:Decision Trees, Random Forest, XGBoost, Logistic Regression
for production systems because they are: Faster, More explainable, Easier to audit, More scalable

KNN is better suited for:Educational projects, Prototypes, Small-scale risk analysis